## Graph API overview

### Graphs
At its core, LangGraph models agent workflows as graphs. You define the behavior of your agents using three key components:

1. **State**: A shared data structure that represents the current snapshot of your application. It can be any data type, but is typically defined using a shared state schema.
2. **Nodes**: Functions that encode the logic of your agents. They receive the current state as input, perform some computation or side-effect, and return an updated state.
3. **Edges**: Functions that determine which Node to execute next based on the current state. They can be conditional branches or fixed transitions.


### StateGraph
The StateGraph class is the main graph class to use. This is parameterized by a user defined State object.

*Compiling your graph*

To build your graph, you first define the state, you then add nodes and edges, and then you compile it. What exactly is compiling your graph and why is it needed?

Compiling is a pretty simple step. It provides a few basic checks on the structure of your graph (no orphaned nodes, etc). It is also where you can specify runtime args like checkpointers and breakpoints. You compile your graph by just calling the .compile method:

```python
graph = graph_builder.compile(...)
```

### state
The first thing you do when you define a graph is define the State of the graph. The State consists of the schema of the graph as well as reducer functions which specify how to apply updates to the state. The schema of the State will be the input schema to all Nodes and Edges in the graph, and can be either a TypedDict or a Pydantic model. All Nodes will emit updates to the State which are then applied using the specified reducer function.

### Schema
The main documented way to specify the schema of a graph is by using a TypedDict. If you want to provide default values in your state, use a dataclass. We also support using a Pydantic BaseModel as your graph state if you want recursive data validation (though note that Pydantic is less performant than a TypedDict or dataclass).

By default, the graph will have the same input and output schemas. If you want to change this, you can also specify explicit input and output schemas directly. This is useful when you have a lot of keys, and some are explicitly for input and others for output. See the guide for more information.


### Multiple schemas
Typically, all graph nodes communicate with a single schema. This means that they will read and write to the same state channels. But, there are cases where we want more control over this:
* Internal nodes can pass information that is not required in the graph’s input / output.
* We may also want to use different input / output schemas for the graph. The output might, for example, only contain a single relevant output key.

It is possible to have nodes write to private state channels inside the graph for internal node communication. We can simply define a private schema, PrivateState.

It is also possible to define explicit input and output schemas for a graph. In these cases, we define an “internal” schema that contains all keys relevant to graph operations. But, we also define input and output schemas that are sub-sets of the “internal” schema to constrain the input and output of the graph. See Define input and output schemas for more detail.

Let’s look at an example:

In [4]:
from typing_extensions import TypedDict


class InputState(TypedDict):
    user_input: str


class OutputState(TypedDict):
    graph_output: str


class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str


class PrivateState(TypedDict):
    bar: str


def node_1(state: InputState) -> OverallState:
    # Write to OverallState
    return {
        "foo": state["user_input"] + " name",
    }


def node_2(state: OverallState) -> PrivateState:
    # Read from OverallState, write to PrivateState
    return {
        "bar": state["foo"] + " is",
    }


def node_3(state: PrivateState) -> OutputState:
    # Read from PrivateState, write to OutputState
    return {"graph_output": state["bar"] + " Nitin."}


from langgraph.graph import StateGraph, START, END

builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
graph.invoke({"user_input": "My"})
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	node_1(node_1)
	node_2(node_2)
	node_3(node_3)
	__end__([<p>__end__</p>]):::last
	__start__ --> node_1;
	node_1 --> node_2;
	node_2 --> node_3;
	node_3 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



There are two subtle and important points to note here:

1. We pass state: InputState as the input schema to node_1. But, we write out to foo, a channel in OverallState. How can we write out to a state channel that is not included in the input schema? This is because a node can write to any state channel in the graph state. The graph state is the union of the state channels defined at initialization, which includes OverallState and the filters InputState and OutputState.
2. We initialize the graph with:
```python
StateGraph(
    OverallState,
    input_schema=InputState,
    output_schema=OutputState
)
```

How can we write to PrivateState in node_2? How does the graph gain access to this schema if it was not passed in the StateGraph initialization?

We can do this because _nodes can also declare additional state channels_ as long as the state schema definition exists. In this case, the PrivateState schema is defined, so we can add bar as a new state channel in the graph and write to it.

### Reducers
Reducers are key to understanding how updates from nodes are applied to the State. Each key in the State has its own independent reducer function. If no reducer function is explicitly specified then it is assumed that all updates to that key should override it. There are a few different types of reducers, starting with the default type of reducer:
​
### Default reducer
These two examples show how to use the default reducer:

In [ ]:
from typing_extensions import TypedDict


class State(TypedDict):
    foo: int
    bar: list[str]

## Functional API overview